# Codveda Technologies — Level 1, Task 1
## Data Preprocessing for Machine Learning

**Dataset:** Telecom Churn (`churn-bigml-80.csv`)

### Objectives
1. Handle missing data
2. Encode categorical variables
3. Normalize/standardize numerical features
4. Split the dataset into training and testing sets

### Professional preprocessing principle
All preprocessing parameters are learned **only from the training data** and then applied to the test data. This prevents **data leakage**.


## 1. Import Required Libraries

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 2. Load the Dataset

The path below works when the dataset is in the same project/data directory as the notebook.  



In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

DATA_PATH = "datasets/churn-bigml-80.csv"

df = pd.read_csv("/content/churn-bigml-80.csv")

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Dataset loaded successfully.
Rows: 2,666
Columns: 20


## 3. Initial Data Inspection

In [ ]:
# ============================================================
# 3. INITIAL DATA INSPECTION
# ============================================================

print("First five rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nColumn data types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


First five rows:


,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   object 
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   object 
 4   Voice mail plan         2666 non-null   object 
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Total night charge

,0
State,object
Account length,int64
Area code,int64
International plan,object
Voice mail plan,object
Number vmail messages,int64
Total day minutes,float64
Total day calls,int64
Total day charge,float64
Total eve minutes,float64



Missing values:


,0
State,0
Account length,0
Area code,0
International plan,0
Voice mail plan,0
Number vmail messages,0
Total day minutes,0
Total day calls,0
Total day charge,0
Total eve minutes,0



Duplicate rows:
0


## 4. Remove Duplicate Records

Duplicate observations can unnecessarily influence preprocessing and later model training, so they are removed before the train/test split.


In [ ]:
# ============================================================
# 4. REMOVE DUPLICATES
# ============================================================

duplicates_before = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print(f"Duplicate rows found: {duplicates_before}")
print(f"Dataset shape after duplicate removal: {df.shape}")


Duplicate rows found: 0
Dataset shape after duplicate removal: (2666, 20)


## 5. Separate Features and Target

`Churn` is the target variable because it represents the outcome we want to predict.

The target is converted from Boolean values (`True`/`False`) to binary values (`1`/`0`). It is **not included in the feature preprocessing pipeline**.


In [ ]:
# ============================================================
# 5. DEFINE FEATURES AND TARGET
# ============================================================

TARGET_COLUMN = "Churn"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].astype(int)

print("Target distribution:")
display(y.value_counts().rename(index={0: "No Churn", 1: "Churn"}))


Target distribution:


,count
Churn,
No Churn,2278
Churn,388


## 6. Identify Numerical and Categorical Features

In [ ]:
# ============================================================
# 6. IDENTIFY FEATURE TYPES
# ============================================================

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)


Categorical features:
['State', 'International plan', 'Voice mail plan']

Numerical features:
['Account length', 'Area code', 'Number vmail messages', 'Total day minutes', 'Total day calls', 'Total day charge', 'Total eve minutes', 'Total eve calls', 'Total eve charge', 'Total night minutes', 'Total night calls', 'Total night charge', 'Total intl minutes', 'Total intl calls', 'Total intl charge', 'Customer service calls']


## 7. Train/Test Split

The split is performed **before fitting any preprocessing transformer**.

A stratified split is used so that the proportion of churned and non-churned customers remains similar in both sets.


In [ ]:
# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training features: {X_train.shape}")
print(f"Testing features:  {X_test.shape}")
print(f"Training target:   {y_train.shape}")
print(f"Testing target:    {y_test.shape}")


Training features: (2132, 19)
Testing features:  (534, 19)
Training target:   (2132,)
Testing target:    (534,)


## 8. Build the Numerical Preprocessing Pipeline

For numerical variables:

- Missing values are replaced using the **median**
- Features are standardized using `StandardScaler`

The imputer is included even if this particular dataset currently contains no missing numerical values. This makes the preprocessing workflow robust and reusable.


In [ ]:
# ============================================================
# 8. NUMERICAL PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


## 9. Build the Categorical Preprocessing Pipeline

For categorical variables:

- Missing values are replaced using the most frequent category
- Categories are converted into numerical columns using **One-Hot Encoding**
- `handle_unknown="ignore"` prevents errors when the test set contains a category not seen during training


In [ ]:
# ============================================================
# 9. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


## 10. Combine the Preprocessing Pipelines

In [ ]:
# ============================================================
# 10. COMBINE PREPROCESSING STEPS
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)


## 11. Fit on Training Data and Transform Both Sets

This is the most important anti-leakage step:

- `fit_transform()` is used **only on training data**
- `transform()` is used on the test data

The test set therefore does not influence the learned imputation, scaling, or encoding parameters.


In [ ]:
# ============================================================
# 11. FIT AND TRANSFORM
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")
print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed testing shape:  {X_test_processed.shape}")


Preprocessing completed successfully.
Processed training shape: (2132, 71)
Processed testing shape:  (534, 71)


## 12. Verify the Transformed Feature Names

In [ ]:
# ============================================================
# 12. FEATURE NAMES AFTER TRANSFORMATION
# ============================================================

feature_names = preprocessor.get_feature_names_out()

processed_preview = pd.DataFrame(
    X_train_processed[:5],
    columns=feature_names
)

display(processed_preview)


,numerical__Account length,numerical__Area code,numerical__Number vmail messages,numerical__Total day minutes,numerical__Total day calls,numerical__Total day charge,numerical__Total eve minutes,numerical__Total eve calls,numerical__Total eve charge,numerical__Total night minutes,...,categorical__State_VA,categorical__State_VT,categorical__State_WA,categorical__State_WI,categorical__State_WV,categorical__State_WY,categorical__International plan_No,categorical__International plan_Yes,categorical__Voice mail plan_No,categorical__Voice mail plan_Yes
0,0.033303,-0.521107,-0.583143,1.688380,-1.009029,1.688657,-0.562170,1.639851,-0.562232,-0.624170,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1.423013,-0.686677,-0.583143,1.118432,-1.655453,1.117947,-0.932862,-0.111558,-0.931779,0.946827,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,-0.547849,-0.521107,-0.583143,-0.655970,-0.511780,-0.656028,-1.268436,-0.461840,-1.269191,0.226294,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,0.791327,-0.686677,-0.583143,-0.178246,0.333544,-0.178628,0.167508,1.039368,0.167680,0.102268,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,-0.143569,-0.686677,-0.583143,1.079698,-1.754903,1.079972,-0.265617,-0.111558,-0.266136,-0.214687,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


## 13. Final Preprocessing Verification

In [ ]:
# ============================================================
# 13. FINAL VERIFICATION
# ============================================================

print("=" * 65)
print("LEVEL 1 - TASK 1: PREPROCESSING SUMMARY")
print("=" * 65)

print("✓ Duplicate observations checked and removed")
print("✓ Missing numerical values handled with median imputation")
print("✓ Missing categorical values handled with most-frequent imputation")
print("✓ Categorical features encoded using One-Hot Encoding")
print("✓ Numerical features standardized using StandardScaler")
print("✓ Data split into training and testing sets")
print("✓ Preprocessing fitted only on training data")
print("✓ Test data transformed without fitting on it")
print("=" * 65)


LEVEL 1 - TASK 1: PREPROCESSING SUMMARY
✓ Duplicate observations checked and removed
✓ Missing numerical values handled with median imputation
✓ Missing categorical values handled with most-frequent imputation
✓ Categorical features encoded using One-Hot Encoding
✓ Numerical features standardized using StandardScaler
✓ Data split into training and testing sets
✓ Preprocessing fitted only on training data
✓ Test data transformed without fitting on it
